In [18]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot
import numpy as np
import os
from datetime import timedelta

pd.set_option('display.max_columns',None)

In [19]:
def get_dataset(dataset:str)->str:
    try:
        return pd.read_csv(os.path.join('..','..','data','raw',f'{dataset}.csv'))
    except Exception as e:
        warnings.warn(f'"{dataset}" dataset not present.')
        return None

In [20]:
def calculate_distance(record):
    lat1 = np.radians( record['customer_latitude'])
    lng1 = np.radians(record['customer_longitude'])
    lat2 = np.radians(record['seller_latitude'])
    lng2 = np.radians(record['seller_longitude'])

    # Calucate the difference.
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    
    a = (np.sin(dlat/2)**2) + np.cos(lat1) * np.cos(lat2) * np.sin(dlng/2)**2

    c =  2 * np.asin(np.sqrt(a))

    r = 6371

    return np.round((c * r),0)
    

In [21]:
data = pd.read_csv(os.path.join('..','..','data','processed','training_data.csv'))

In [22]:
data.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_latitude,customer_longitude,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,seller_latitude,seller_longitude
0,18668c1a8328478efec728fc614c3b55,6d547f62fd060718fd2a268a4fa2a1fa,delivered,2017-05-04 15:06:03,2017-05-04 15:22:30,2017-05-05 12:00:40,2017-05-23 11:12:49,2017-05-25,1.0,credit_card,8.0,162.10,288bca9dae3585557dfd5dfdf7289600,9371,maua,SP,-23.669305,-46.477297,1.0,4fcb3d9a5f4871e8362dfedbdb02b064,8581055ce74af1daba164fdbd55a40de,2017-05-10 15:22:30,143.80,18.30,automotivo,53.0,603.0,1.0,5950.0,65.0,11.0,65.0,auto,7112.0,guarulhos,SP,-23.468737,-46.513845
1,e35bd284b205ea082108ae3703c84428,ac8e62b68c0a8fd4d4eb818903300c68,delivered,2017-11-10 13:42:22,2017-11-13 07:50:30,2017-11-14 16:09:51,2017-11-22 14:12:32,2017-12-05,1.0,credit_card,4.0,354.16,9b85684cedccb35b49cbad4fcfe04417,4119,sao paulo,SP,-23.592552,-46.634911,1.0,52c80cedd4e90108bf4fa6a206ef6b03,a1043bafd471dff536d0c462352beb48,2017-11-20 07:50:30,179.99,46.44,ferramentas_jardim,33.0,2188.0,2.0,7650.0,20.0,20.0,20.0,garden_tools,37175.0,ilicinea,MG,-20.940628,-45.826803
2,2fd573eea535b447eea60a9b4aa2603d,f94bd0795b4e39a9250f516c56ee4309,delivered,2018-08-23 17:22:26,2018-08-23 17:35:14,2018-08-24 10:38:00,2018-08-29 15:28:56,2018-09-04,1.0,credit_card,3.0,243.51,7e706e55aec816de5d9b8b8799f5aefd,37701,pocos de caldas,MG,-21.788130,-46.565546,1.0,7c88ed4fc8c3ae3a1e08a074cbfc9a88,0bebbb2cea103a4a020c95d43fd7d754,2018-08-27 17:31:06,217.90,25.61,beleza_saude,58.0,1348.0,1.0,1317.0,22.0,28.0,17.0,health_beauty,2515.0,sao paulo,SP,-23.510254,-46.657770


In [35]:
data.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date        1
order_delivered_customer_date       7
order_estimated_delivery_date       0
payment_sequential                  0
payment_type                        0
payment_installments                0
payment_value                       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
customer_latitude                   0
customer_longitude                  0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name            1181
product_name_lenght              1181
product_desc

In [25]:
# Convert columns to their appropriate datatypes.

date_col = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
            'order_delivered_customer_date','order_estimated_delivery_date','shipping_limit_date']

number_col = ['payment_sequential','payment_installments','order_item_id','seller_zip_code_prefix']

for col in date_col:
    data[col] = pd.to_datetime( data[col])
    

for col in number_col:
    data[col] = data[col].astype('Int64')



In [26]:
data = data[data['order_status'] == 'delivered']

In [28]:
df_geolocation = get_dataset('geolocation')

In [29]:
df_unique_geolocation = df_geolocation.groupby('geolocation_zip_code_prefix').agg(latitude = ('geolocation_lat','median'),longitude=('geolocation_lng','median')).reset_index()

In [30]:
data[data['customer_latitude'].isna()]['customer_zip_code_prefix'].isin(df_unique_geolocation['geolocation_zip_code_prefix']).sum()

np.int64(0)

In [31]:
data[data['seller_latitude'].isna()]['seller_zip_code_prefix'].isin(df_unique_geolocation['geolocation_zip_code_prefix']).sum()

np.int64(0)

In [32]:
# Those zip code are not present in geolocation table so we need to drop them.

data.dropna(axis=0,subset=['customer_latitude','customer_longitude','seller_latitude','seller_longitude'],inplace=True)

In [34]:
# to order_approved at null values filling , first we take the mean time of order approved from order purchased which is '10 hourse' 
# then we fill the null values using order_purchase_time + 10 hours (which is aproximate)




duration = timedelta(hours=10)
data['order_approved_at'] = data['order_approved_at'].fillna(value= (data['order_purchase_timestamp']+duration))

In [15]:
# the remaining null values count are very low so we drop them directly before droping '82221 records'

data.dropna(inplace=True)


In [16]:
data['delivery_distance_km']  = data.apply(calculate_distance, axis=1)

In [19]:
data.to_csv(os.path.join('..','..','data','processed','cleaned_trained_data.csv'),index=False)

In [20]:
os.getcwd()

'D:\\Machine Learning\\projects\\E_commerce\\notebooks\\data working steps'

In [21]:
data.info()

<class 'pandas.DataFrame'>
Index: 81048 entries, 0 to 85067
Data columns (total 39 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       81048 non-null  str           
 1   customer_id                    81048 non-null  str           
 2   order_status                   81048 non-null  str           
 3   order_purchase_timestamp       81048 non-null  datetime64[us]
 4   order_approved_at              81048 non-null  datetime64[us]
 5   order_delivered_carrier_date   81048 non-null  datetime64[us]
 6   order_delivered_customer_date  81048 non-null  datetime64[us]
 7   order_estimated_delivery_date  81048 non-null  datetime64[us]
 8   payment_sequential             81048 non-null  Int64         
 9   payment_type                   81048 non-null  str           
 10  payment_installments           81048 non-null  Int64         
 11  payment_value                  

In [36]:
imp_features = ['order_id','order_purchase_timestamp','order_delivered_customer_date',
                'payment_type','customer_state','customer_latitude',
                'customer_longitude','price','freight_value','product_weight_g',
                'seller_state','seller_latitude','seller_longitude']

In [37]:
data[imp_features]

,order_id,order_purchase_timestamp,order_delivered_customer_date,payment_type,customer_state,customer_latitude,customer_longitude,price,freight_value,product_weight_g,seller_state,seller_latitude,seller_longitude
0,18668c1a8328478efec728fc614c3b55,2017-05-04 15:06:03,2017-05-23 11:12:49,credit_card,SP,-23.669305,-46.477297,143.80,18.30,5950.0,SP,-23.468737,-46.513845
1,e35bd284b205ea082108ae3703c84428,2017-11-10 13:42:22,2017-11-22 14:12:32,credit_card,SP,-23.592552,-46.634911,179.99,46.44,7650.0,MG,-20.940628,-45.826803
2,2fd573eea535b447eea60a9b4aa2603d,2018-08-23 17:22:26,2018-08-29 15:28:56,credit_card,MG,-21.788130,-46.565546,217.90,25.61,1317.0,SP,-23.510254,-46.657770
3,73c8ab38f07dc94389065f7eba4f297a,2017-12-13 14:21:15,2017-12-28 09:05:34,boleto,SP,-23.965875,-46.341508,59.00,13.43,1550.0,SP,-20.806308,-49.388423
4,26f35d2f228f74c5c9ca3a74c3a88d7c,2017-06-11 13:38:21,2017-06-16 12:05:06,credit_card,SP,-22.940480,-47.042770,28.99,13.08,2600.0,SP,-23.467754,-46.551961
...,...,...,...,...,...,...,...,...,...,...,...,...,...
85063,c0c735876359d7431dbd45a8fb0e238b,2018-02-23 00:11:32,2018-03-08 15:24:09,credit_card,PB,-7.103643,-34.856495,32.99,24.90,1550.0,SP,-23.591193,-46.631725
85064,24940d1052df86d8e24f16e5adf2868e,2017-05-01 13:31:40,2017-05-16 20:21:26,credit_card,SP,-22.754099,-47.621645,174.90,12.61,1600.0,SP,-23.568488,-46.522269
85065,559d6842e0a3468f1941683ca901b582,2018-03-12 22:25:18,2018-04-24 21:39:33,credit_card,MT,-13.070557,-55.918909,569.00,25.69,425.0,SP,-21.171135,-47.822761
85066,1146fb180e28dfbfdbfabd7249026429,2018-02-23 11:04:11,2018-03-03 14:04:16,credit_card,SP,-20.540165,-48.558852,39.90,13.37,1300.0,SP,-23.533770,-47.468661
